# RQ1 — Per-scheme decomposition with exact 8! node-permutation test

Successor to [rq1_perscheme_decomposition.ipynb](rq1_perscheme_decomposition.ipynb)
that recomputes p-values using **exact node permutation** (Mantel/QAP style)
instead of edge-wise cosine shuffling.

## Why this matters

The 28 dyads share behavior nodes — with 8 behaviors, each appears in 7 pairs.
Edge-wise cos shuffling treats rows as exchangeable, which is the dependence
structure we need to *defend against*. Node permutation respects the dyadic
dependence: we relabel the 8 behaviors via a permutation π, apply π to both
rows and columns of the cosine matrix simultaneously, keep everything else
fixed at its observed (i, j) positions, and refit.

With 8 behaviors, **8! = 40,320** permutations is exhaustive — the resulting
null distribution is *exact*, not Monte-Carlo. The β estimates are the same as
the existing per-scheme analysis (we don't change the model); only the
p-values change.

## Setup (verbatim from the per-scheme notebook)

- Frame: `analysis/rq1_consolidated/consolidated_coh30.csv`, coh ≥ 30
  (row-level).
- Two schemes, each fit separately: v2 / normTrue (n=28) and v3 / per_axis
  (n=22 after the 6-pair coh drop).
- Model: `y ~ mean_single_abs + max_single_abs + cos`, standardized.
- No `mechanical_push` (collinear with cos within a single scheme) and no
  scheme dummy (single scheme per fit).
- Outcomes: `mean_joint_abs` (magnitude) and `supp_mean_signed` (direction).
- Robustness: + sem_sim, |cos|, leave-one-trait-out — all re-tested with the
  node-perm p-value.

## Node-perm test recipe (from the brief)

For each scheme separately:
1. Build 8×8 symmetric matrices for cos, Y (each outcome), mean_single_abs,
   max_single_abs, and the coh mask.
2. Apply the coh mask (per-axis: 6 cells masked out; normTrue: none).
3. Fit the observed standardized regression y ~ mean_single + max_single + cos
   on masked upper-triangle cells; get β_cos^obs.
4. For each of the 8! = 40,320 behavior relabelings π, compute
   `cos_perm = cos[π, π]` (relabel rows and cols together), extract the same
   masked upper-triangle cells, refit, get β_cos^π. Y, mean_single, max_single,
   and the mask all stay fixed at their original (i, j) positions.
5. Exact two-sided p = fraction of |β_cos^π| ≥ |β_cos^obs|.

In [1]:
# Imports + paths + load
from pathlib import Path
from itertools import permutations
import warnings
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")

REPO = Path.cwd().parents[1]
CSV_PATH = REPO / "analysis/rq1_consolidated/consolidated_coh30.csv"

df_all = pd.read_csv(CSV_PATH)
df_all["supp_signed_a"]    = df_all["delta_a_joint"] - df_all["delta_a_single"]
df_all["supp_signed_b"]    = df_all["delta_b_joint"] - df_all["delta_b_single"]
df_all["supp_mean_signed"] = 0.5 * (df_all["supp_signed_a"] + df_all["supp_signed_b"])

df_v2 = df_all[df_all["scheme"] == "normTrue"].reset_index(drop=True)
df_v3 = df_all[df_all["scheme"] == "per_axis"].reset_index(drop=True)

# IMPORTANT: trait set is PER SCHEME. v2 and v3 use the 8-trait set
# (power_seeking was dropped in Phase 12.5). The pooled union would be 9, but
# permuting power_seeking into v2/v3 cells produces NaN cosines and silently
# biases the p-value toward significance. We use the 8 traits actually present
# in each scheme — for v2 and v3 these are identical sets.
TRAITS_v2 = sorted(set(df_v2["trait_a"]).union(df_v2["trait_b"]))
TRAITS_v3 = sorted(set(df_v3["trait_a"]).union(df_v3["trait_b"]))
assert TRAITS_v2 == TRAITS_v3, "v2 and v3 should share the 8-trait set"
TRAITS = TRAITS_v2
n_traits = len(TRAITS)
print(f"Per-scheme trait set ({n_traits}): {TRAITS}")
print(f"v2 / normTrue: n = {len(df_v2)}")
print(f"v3 / per_axis: n = {len(df_v3)}")
print(f"Permutation count: {n_traits}! = {np.math.factorial(n_traits):,}")

Per-scheme trait set (8): ['apathetic', 'confidence', 'evil', 'formality', 'hallucinating', 'humorous', 'impolite', 'sycophantic']
v2 / normTrue: n = 28
v3 / per_axis: n = 22
Permutation count: 8! = 40,320


In [2]:
# Build 8x8 matrices and coh mask from the long-form per-scheme frame
def build_matrices(sub, traits=TRAITS):
    n = len(traits)
    idx = {t: i for i, t in enumerate(traits)}
    M_cos = np.full((n, n), np.nan)
    M_mag = np.full((n, n), np.nan)
    M_dir = np.full((n, n), np.nan)
    M_ms  = np.full((n, n), np.nan)   # mean_single_abs
    M_Ms  = np.full((n, n), np.nan)   # max_single_abs
    M_sem = np.full((n, n), np.nan)   # sem_sim (for robustness)
    mask  = np.zeros((n, n), bool)
    for _, row in sub.iterrows():
        i, j = idx[row.trait_a], idx[row.trait_b]
        for (a, b) in [(i, j), (j, i)]:
            M_cos[a, b] = row["cos"]
            M_mag[a, b] = row["mean_joint_abs"]
            M_dir[a, b] = row["supp_mean_signed"]
            M_ms[a, b]  = row["mean_single_abs"]
            M_Ms[a, b]  = row["max_single_abs"]
            M_sem[a, b] = row["sem_sim"]
            mask[a, b]  = True
    return dict(cos=M_cos, mag=M_mag, dir=M_dir,
                ms=M_ms, Ms=M_Ms, sem=M_sem, mask=mask)


mats_v2 = build_matrices(df_v2)
mats_v3 = build_matrices(df_v3)

iu = np.triu_indices(n_traits, k=1)
print(f"Upper-triangle dyads available: {iu[0].size}")
print(f"v2 masked-in dyads: {mats_v2['mask'][iu].sum()}  (should equal {len(df_v2)})")
print(f"v3 masked-in dyads: {mats_v3['mask'][iu].sum()}  (should equal {len(df_v3)})")
assert mats_v2['mask'][iu].sum() == len(df_v2)
assert mats_v3['mask'][iu].sum() == len(df_v3)

Upper-triangle dyads available: 28
v2 masked-in dyads: 28  (should equal 28)
v3 masked-in dyads: 22  (should equal 22)


In [3]:
# Standardized OLS on the masked upper-triangle dyads; returns β_cos (last col).
def fit_beta_cos(cos_v, ms_v, Ms_v, y_v):
    """Standardize all predictors and outcome, fit y ~ ms + Ms + cos, return β_cos."""
    def z(a):
        s = a.std(ddof=0)
        return (a - a.mean()) / s if s > 0 else np.zeros_like(a)
    X = np.column_stack([np.ones(len(y_v)), z(ms_v), z(Ms_v), z(cos_v)])
    y_z = z(y_v)
    beta, *_ = np.linalg.lstsq(X, y_z, rcond=None)
    return float(beta[3])


def extract(M, mask_vec, perm=None):
    """Upper-triangle values at masked-in cells; M optionally relabeled by perm."""
    if perm is not None:
        M = M[np.ix_(perm, perm)]
    return M[iu][mask_vec]


def node_perm_p(mats, y_key, extra_predictors=None, return_null=False):
    """Exact 8! node-permutation test on β_cos.

    extra_predictors: dict of name -> matrix to add as additional fixed
    (non-permuted) controls. Used for the + sem_sim robustness probe."""
    mask_vec = mats["mask"][iu]
    ms_obs   = extract(mats["ms"], mask_vec)
    Ms_obs   = extract(mats["Ms"], mask_vec)
    y_obs    = extract(mats[y_key], mask_vec)
    cos_obs  = extract(mats["cos"], mask_vec)

    if extra_predictors is None:
        # Plain y ~ ms + Ms + cos
        b_obs = fit_beta_cos(cos_obs, ms_obs, Ms_obs, y_obs)
        null = np.empty(np.math.factorial(n_traits))
        for k, perm in enumerate(permutations(range(n_traits))):
            cos_p = extract(mats["cos"], mask_vec, perm=np.array(perm))
            null[k] = fit_beta_cos(cos_p, ms_obs, Ms_obs, y_obs)
    else:
        # General case: add one extra control (e.g. sem_sim) to the design
        extras = [extract(M, mask_vec) for M in extra_predictors.values()]
        def fit_with_extras(cos_v):
            def z(a):
                s = a.std(ddof=0)
                return (a - a.mean()) / s if s > 0 else np.zeros_like(a)
            cols = [z(ms_obs), z(Ms_obs)] + [z(e) for e in extras] + [z(cos_v)]
            X = np.column_stack([np.ones(len(y_obs))] + cols)
            y_z = z(y_obs)
            beta, *_ = np.linalg.lstsq(X, y_z, rcond=None)
            return float(beta[-1])
        b_obs = fit_with_extras(cos_obs)
        null = np.empty(np.math.factorial(n_traits))
        for k, perm in enumerate(permutations(range(n_traits))):
            cos_p = extract(mats["cos"], mask_vec, perm=np.array(perm))
            null[k] = fit_with_extras(cos_p)

    p_two = float(np.mean(np.abs(null) >= np.abs(b_obs)))
    if return_null:
        return b_obs, p_two, null
    return b_obs, p_two


print("Utilities loaded.")

Utilities loaded.


## §1 — Main result with node-perm p-values

The β estimates are unchanged from the existing per-scheme analysis. Only the
p-values are recomputed (now exact, not Monte-Carlo).

In [4]:
print("=== Per-scheme decomposition with exact 8! node-permutation p ===\n")

rows = []
for label, mats in [("v2 / normTrue", mats_v2), ("v3 / per_axis", mats_v3)]:
    for outcome_name, y_key in [("MAGNITUDE (mean_joint_abs)", "mag"),
                                  ("DIRECTION (supp_mean_signed)", "dir")]:
        b_obs, p_node, null = node_perm_p(mats, y_key, return_null=True)
        # Compare to the bivariate r for context
        mask_vec = mats["mask"][iu]
        cos_obs = extract(mats["cos"], mask_vec)
        y_obs   = extract(mats[y_key], mask_vec)
        r_biv, p_biv = stats.pearsonr(cos_obs, y_obs)
        rows.append({"scheme": label, "outcome": outcome_name,
                     "n_dyads": int(mask_vec.sum()),
                     "β_cos (std)": b_obs,
                     "node-perm p (8!)": p_node,
                     "r(cos, y) biv": r_biv, "p biv": p_biv,
                     "n_perms": len(null)})

tbl = pd.DataFrame(rows)[["scheme","outcome","n_dyads","β_cos (std)","node-perm p (8!)","r(cos, y) biv","p biv","n_perms"]]
print(tbl.round(4).to_string(index=False))

=== Per-scheme decomposition with exact 8! node-permutation p ===



       scheme                      outcome  n_dyads  β_cos (std)  node-perm p (8!)  r(cos, y) biv  p biv  n_perms
v2 / normTrue   MAGNITUDE (mean_joint_abs)       28       0.1882            0.0808         0.6459 0.0002    40320
v2 / normTrue DIRECTION (supp_mean_signed)       28       0.3096            0.0470         0.4754 0.0106    40320
v3 / per_axis   MAGNITUDE (mean_joint_abs)       22       0.1868            0.0001         0.6421 0.0013    40320
v3 / per_axis DIRECTION (supp_mean_signed)       22       0.4105            0.0001         0.4724 0.0264    40320


## §2 — Side-by-side: edge-wise vs node-permutation p-values

The β values are identical (same fit). What changes is the p-value: edge-wise
shuffling gave us p ≈ 0.04 on both directional cells; the node-perm test is
the dyadic null we actually need.

In [5]:
# Edge-wise p-values from the prior per-scheme notebook (recomputed here so the comparison is hermetic)
def edge_perm_p(sub, y_col, n_perm=10_000, rng=None):
    rng = rng or np.random.default_rng(0)
    def beta_cos_std(sub_loc):
        def z(a):
            a = np.asarray(a, float); s = a.std(ddof=0)
            return (a - a.mean()) / s if s > 0 else np.zeros_like(a)
        cols = ["mean_single_abs", "max_single_abs", "cos"]
        X = np.column_stack([np.ones(len(sub_loc)), z(sub_loc["mean_single_abs"]),
                             z(sub_loc["max_single_abs"]), z(sub_loc["cos"])])
        y_z = z(sub_loc[y_col].values)
        beta, *_ = np.linalg.lstsq(X, y_z, rcond=None)
        return float(beta[3])
    obs = beta_cos_std(sub)
    sub_loc = sub.copy()
    cos_arr = sub_loc["cos"].values.copy()
    null = np.empty(n_perm)
    for k in range(n_perm):
        rng.shuffle(cos_arr)
        sub_loc["cos"] = cos_arr
        null[k] = beta_cos_std(sub_loc)
    return obs, float(np.mean(np.abs(null) >= np.abs(obs)))


print("=== Edge-wise (10k) vs Node-perm (exact 8!) p-values ===\n")
rows = []
for label, mats, sub in [("v2 / normTrue", mats_v2, df_v2),
                          ("v3 / per_axis", mats_v3, df_v3)]:
    for outcome_name, y_key, y_col in [
        ("MAGNITUDE", "mag", "mean_joint_abs"),
        ("DIRECTION", "dir", "supp_mean_signed")]:
        b_edge, p_edge = edge_perm_p(sub, y_col)
        b_node, p_node = node_perm_p(mats, y_key)
        assert abs(b_edge - b_node) < 1e-9, "β estimates should be identical"
        rows.append({"scheme": label, "outcome": outcome_name,
                     "β_cos (std)": b_node,
                     "edge p (10k)": p_edge,
                     "node-perm p (8!)": p_node,
                     "difference": p_node - p_edge})
print(pd.DataFrame(rows).round(4).to_string(index=False))

=== Edge-wise (10k) vs Node-perm (exact 8!) p-values ===



       scheme   outcome  β_cos (std)  edge p (10k)  node-perm p (8!)  difference
v2 / normTrue MAGNITUDE       0.1882        0.0676            0.0808      0.0132
v2 / normTrue DIRECTION       0.3096        0.0416            0.0470      0.0054
v3 / per_axis MAGNITUDE       0.1868        0.1477            0.0001     -0.1476
v3 / per_axis DIRECTION       0.4105        0.0469            0.0001     -0.0468


## §3 — Robustness with node-perm p-values

Re-run the three direction robustness checks under the dyadic null:
- + sem_sim
- |cos| instead of signed cos
- Leave-one-behavior-out

LOO note: when we drop a trait, the matrix shrinks from 8×8 to 7×7 and the
permutation count drops from 8! = 40,320 to 7! = 5,040. Both are exhaustive.

In [6]:
# 3a. + sem_sim with node-perm p
print("=== 3a. + sem_sim control (directional, node-perm p) ===\n")
for label, mats in [("v2 / normTrue", mats_v2), ("v3 / per_axis", mats_v3)]:
    b_obs, p_node = node_perm_p(mats, "dir", extra_predictors={"sem": mats["sem"]})
    print(f"  {label}: β_cos = {b_obs:+.4f}  node-perm p = {p_node:.4f}")

=== 3a. + sem_sim control (directional, node-perm p) ===



  v2 / normTrue: β_cos = +0.3614  node-perm p = 0.0231


  v3 / per_axis: β_cos = +0.4433  node-perm p = 0.0001


In [7]:
# 3b. |cos| substituted into the cos slot — node-perm p
def node_perm_p_abscos(mats, y_key):
    """Substitute |cos| matrix for the cos matrix; everything else as in node_perm_p."""
    mats_abs = dict(mats)
    mats_abs["cos"] = np.abs(mats["cos"])
    return node_perm_p(mats_abs, y_key)


print("=== 3b. |cos| substituted for signed cos (directional, node-perm p) ===\n")
for label, mats in [("v2 / normTrue", mats_v2), ("v3 / per_axis", mats_v3)]:
    b_signed, p_signed = node_perm_p(mats, "dir")
    b_abs, p_abs       = node_perm_p_abscos(mats, "dir")
    print(f"  {label}")
    print(f"    signed cos: β = {b_signed:+.4f}  node-perm p = {p_signed:.4f}")
    print(f"    |cos|:      β = {b_abs:+.4f}  node-perm p = {p_abs:.4f}")
    print()

=== 3b. |cos| substituted for signed cos (directional, node-perm p) ===



  v2 / normTrue
    signed cos: β = +0.3096  node-perm p = 0.0470
    |cos|:      β = +0.0527  node-perm p = 0.7416



  v3 / per_axis
    signed cos: β = +0.4105  node-perm p = 0.0001
    |cos|:      β = +0.1931  node-perm p = 0.0002



In [8]:
# 3c. Leave-one-trait-out with node-perm p (each fold uses 7! = 5040 permutations)
def loo_node_perm(sub, traits_full=TRAITS):
    rows = []
    # Full fit first (recompute β for sanity)
    mats_full = build_matrices(sub, traits_full)
    b0, p0 = node_perm_p(mats_full, "dir")
    rows.append({"drop": "(none)", "n_traits": len(traits_full),
                 "n_dyads": int(mats_full["mask"][iu].sum()),
                 "β_cos": b0, "node-perm p": p0})
    # Per-trait drops
    for drop in traits_full:
        keep = [t for t in traits_full if t != drop]
        sub_loo = sub[(sub["trait_a"].isin(keep)) & (sub["trait_b"].isin(keep))]
        mats_loo = build_matrices(sub_loo, keep)
        nk = len(keep)
        iu_loo = np.triu_indices(nk, k=1)
        mask_vec_loo = mats_loo["mask"][iu_loo]
        cos_obs = mats_loo["cos"][iu_loo][mask_vec_loo]
        ms_obs  = mats_loo["ms"][iu_loo][mask_vec_loo]
        Ms_obs  = mats_loo["Ms"][iu_loo][mask_vec_loo]
        y_obs   = mats_loo["dir"][iu_loo][mask_vec_loo]
        if len(y_obs) < 5:
            rows.append({"drop": drop, "n_traits": nk, "n_dyads": len(y_obs),
                         "β_cos": np.nan, "node-perm p": np.nan})
            continue
        b_obs = fit_beta_cos(cos_obs, ms_obs, Ms_obs, y_obs)
        null = np.empty(np.math.factorial(nk))
        for k, perm in enumerate(permutations(range(nk))):
            cos_p = mats_loo["cos"][np.ix_(perm, perm)][iu_loo][mask_vec_loo]
            null[k] = fit_beta_cos(cos_p, ms_obs, Ms_obs, y_obs)
        p_node = float(np.mean(np.abs(null) >= np.abs(b_obs)))
        rows.append({"drop": drop, "n_traits": nk, "n_dyads": int(mask_vec_loo.sum()),
                     "β_cos": b_obs, "node-perm p": p_node})
    return pd.DataFrame(rows)


print("=== 3c. LOO with node-perm p (directional) ===\n")
print("--- v2 / normTrue ---")
loo_v2 = loo_node_perm(df_v2)
print(loo_v2.round(4).to_string(index=False))
print()
print("--- v3 / per_axis ---")
loo_v3 = loo_node_perm(df_v3)
print(loo_v3.round(4).to_string(index=False))

=== 3c. LOO with node-perm p (directional) ===

--- v2 / normTrue ---


         drop  n_traits  n_dyads  β_cos  node-perm p
       (none)         8       28 0.3096       0.0470
    apathetic         7       21 0.0636       0.7540
   confidence         7       21 0.4053       0.0286
         evil         7       21 0.4183       0.0127
    formality         7       21 0.2632       0.2423
hallucinating         7       21 0.6317       0.0002
     humorous         7       21 0.2877       0.1351
     impolite         7       21 0.3345       0.0875
  sycophantic         7       21 0.2132       0.2159

--- v3 / per_axis ---


         drop  n_traits  n_dyads  β_cos  node-perm p
       (none)         8       22 0.4105       0.0001
    apathetic         7       16 0.2462       0.0004
   confidence         7       16 0.5186       0.0004
         evil         7       16 0.4632       0.0002
    formality         7       16 0.3837       0.0006
hallucinating         7       18 0.5487       0.0004
     humorous         7       19 0.3819       0.0024
     impolite         7       16 0.2693       0.0006
  sycophantic         7       15 0.4304       0.0008


## §4 — Final printable block

Numbers in this block are what should go in the main-paper table. The exact
node-perm p replaces the edge-shuffle p; the β estimates are identical to the
existing per-scheme analysis.

In [9]:
print("=" * 72)
print("FINAL TABLE (paper-ready) — per-scheme, exact 8! node-permutation")
print("=" * 72)
print()
print("Frame: consolidated_coh30 (coh ≥ 30 row-level)")
print(f"v2 / normTrue: n = {len(df_v2)} dyads (0 dropped by coh filter)")
print(f"v3 / per_axis: n = {len(df_v3)} dyads (6 dropped by coh filter)")
print()
print("Model: y ~ mean_single_abs + max_single_abs + cos (all standardized)")
print(f"Null: exact 8! = {np.math.factorial(n_traits):,} node permutations of the cosine matrix")
print()
for label, mats in [("v2 / normTrue", mats_v2), ("v3 / per_axis", mats_v3)]:
    print(f"--- {label} ---")
    for outcome_name, y_key in [("MAGNITUDE (mean_joint_abs)",   "mag"),
                                  ("DIRECTION (supp_mean_signed)", "dir")]:
        b, p = node_perm_p(mats, y_key)
        print(f"  {outcome_name:32s}  β_cos = {b:+.4f}  node-perm p = {p:.4f}")
    print()
print("=" * 72)

FINAL TABLE (paper-ready) — per-scheme, exact 8! node-permutation

Frame: consolidated_coh30 (coh ≥ 30 row-level)
v2 / normTrue: n = 28 dyads (0 dropped by coh filter)
v3 / per_axis: n = 22 dyads (6 dropped by coh filter)

Model: y ~ mean_single_abs + max_single_abs + cos (all standardized)
Null: exact 8! = 40,320 node permutations of the cosine matrix

--- v2 / normTrue ---


  MAGNITUDE (mean_joint_abs)        β_cos = +0.1882  node-perm p = 0.0808


  DIRECTION (supp_mean_signed)      β_cos = +0.3096  node-perm p = 0.0470

--- v3 / per_axis ---


  MAGNITUDE (mean_joint_abs)        β_cos = +0.1868  node-perm p = 0.0001


  DIRECTION (supp_mean_signed)      β_cos = +0.4105  node-perm p = 0.0001

